In [1]:
using Lux, Reactant, Enzyme, MLUtils, NNlib
using Optimisers, Random, Statistics, Images
using LinearAlgebra, Images, JLD2, ComponentArrays
using Dates, Plots

In [2]:
using MLDatasets
imgs_color, _ = CIFAR10(split=:train)[:]
imgs = reshape(mean(Float32.(imgs_color), dims=3),(32,32,50000))

32×32×50000 Array{Float32, 3}:
[:, :, 1] =
 0.240523  0.0732026  0.0915033  0.0980392  …  0.619608  0.542484   0.571242
 0.175163  0.0        0.0300654  0.0810458     0.50719   0.44183    0.511111
 0.184314  0.0339869  0.109804   0.216993      0.503268  0.470588   0.533333
 0.214379  0.112418   0.203922   0.257516      0.490196  0.486275   0.528105
 0.291503  0.20915    0.291503   0.284967      0.47451   0.509804   0.572549
 0.356863  0.320261   0.359477   0.295425   …  0.44183   0.524183   0.631373
 0.419608  0.342484   0.363399   0.291503      0.420915  0.488889   0.628758
 0.437909  0.335948   0.326797   0.261438      0.4       0.447059   0.598693
 0.464052  0.343791   0.304575   0.271895      0.364706  0.407843   0.566013
 0.473203  0.312418   0.305882   0.326797      0.343791  0.360784   0.522876
 ⋮                                          ⋱            ⋮          
 0.423529  0.322876   0.350327   0.360784      0.328105  0.278431   0.350327
 0.471895  0.309804   0.345098   0.332026

In [3]:
spatial_conv_layer = Conv((17, 17), 2 => 256; pad = 8)
struct ConvFirstTwo{C}<: Lux.AbstractLuxWrapperLayer{:conv}
    conv::C
    length::Int
end
# 2. Explicitly define how state is generated
function Lux.initialstates(rng::AbstractRNG, m::ConvFirstTwo)
    return (conv = Lux.initialstates(rng, m.conv),)
end

# 3. Explicitly define how parameters are generated
function Lux.initialparameters(rng::AbstractRNG, m::ConvFirstTwo)
    return (conv = Lux.initialparameters(rng, m.conv),)
end
function (m::ConvFirstTwo)(x, ps, st) 
    # Extract the slice
    x1 = x[:, :, 1:2, :]
    x2 = x[:, :, 3:m.length, :]

    # IMPORTANT: Explicitly pass m.conv its specific inner ps and st
    y1, st_conv = m.conv(x1, ps.conv, st.conv)

    # Return the concatenated outputs alongside the updated state structure
    return cat(y1, x2; dims=3), (conv = st_conv,)
end

first_conv = ConvFirstTwo(spatial_conv_layer, 34)

ConvFirstTwo(
    conv = Conv((17, 17), 2 => 256, pad=8),       # 148_224 parameters
)         # Total: 148_224 parameters,
          #        plus 0 states.

In [4]:
emb_dim = 32
model= Chain(
    #Conv((17, 17), 2 => 256; pad = 8),
    first_conv,
    #last_conv,
    SkipConnection(
        Chain(
            Conv((1, 1), 256+emb_dim => 512, gelu),
            SkipConnection(
                Chain(
                    Conv((1, 1), 512 => 512, gelu),
                    Conv((1, 1), 512 => 512, gelu),
                    Conv((1, 1), 512 => 512, gelu),
                ),
                +
            ),
            Conv((1, 1), 512 => 256, gelu),
            Conv((1, 1), 256 => 256+emb_dim, gelu),
        ),
        +
    ),
    Conv((1, 1), 256+emb_dim => 128, gelu),
    Conv((1, 1), 128 => 64, gelu),
    Conv((1, 1), 64 => 1)
)

Chain(
    layer_1 = ConvFirstTwo(
        conv = Conv((17, 17), 2 => 256, pad=8),   # 148_224 parameters
    ),
    layer_2 = SkipConnection(
        connection = +,
        layers = Chain(
            layer_1 = Conv((1, 1), 288 => 512, gelu_tanh),  # 147_968 parameters
            layer_2 = SkipConnection(
                connection = +,
                layers = Chain(
                    layer_(1-3) = Conv((1, 1), 512 => 512, gelu_tanh),  # 787_968 (262_656 x 3) parameters
                ),
            ),
            layer_3 = Conv((1, 1), 512 => 256, gelu_tanh),  # 131_328 parameters
            layer_4 = Conv((1, 1), 256 => 288, gelu_tanh),  # 74_016 parameters
        ),
    ),
    layer_3 = Conv((1, 1), 288 => 128, gelu_tanh),  # 36_992 parameters
    layer_4 = Conv((1, 1), 128 => 64, gelu_tanh),  # 8_256 parameters
    layer_5 = Conv((1, 1), 64 => 1),              # 65 parameters
)         # Total: 1_334_817 parameters,
          #        plus 0 states.

In [5]:
emb_dim = 32
#include("model.jl")

32

In [6]:
#emb_dim = num_sin_channels
load_model = false
test_model = false
#include("model.jl"); emb_dim = 0
include("compiler.jl")

I0000 00:00:1780752658.670029   40347 service.cc:178] XLA service 0x27ffea30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780752658.670047   40347 service.cc:194]   StreamExecutor [0]: NVIDIA GeForce RTX 3080, Compute Capability 8.6 (Driver: 13.3.0[610.43.2]; Runtime: 13.1.0; Toolkit: 13.1.0; DNN: 9.14.0)
I0000 00:00:1780752658.670314   40347 se_gpu_pjrt_client.cc:1520] Using BFC allocator.
I0000 00:00:1780752658.670336   40347 gpu_helpers.cc:141] XLA backend allocating 7763607552 bytes on device 0 for BFCAllocator.
I0000 00:00:1780752658.670356   40347 gpu_helpers.cc:183] XLA backend will use up to 2587869184 bytes on device 0 for CollectiveBFCAllocator.
I0000 00:00:1780752658.673145   40347 cuda_dnn.cc:461] Loaded cuDNN version 91400


In [7]:
im_path = "../../../Images/64/"
# Hyperparameters for Variance Preserving (VP) SDE
const βmin = 0.1
const βmax = 20.0
T = 1

1

In [8]:
include("helperfuncs.jl")

load_images (generic function with 2 methods)

In [9]:
x_img, y_img = load_images_to_array(im_path,1f-1 .* rand(Float32, 100),100,forward, 64,32)

(Float32[0.5298159 0.9019003 … 0.8515273 0.70373607; 1.2002103 1.0993451 … 0.30907744 0.5669468; … ; -0.11269882 1.2696029 … 0.16965073 -0.039090622; 0.55418396 1.0283636 … 0.020703657 0.25772506;;; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0; … ; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0;;; 0.096988454 0.096988454 … 0.096988454 0.096988454; 0.096988454 0.096988454 … 0.096988454 0.096988454; … ; 0.096988454 0.096988454 … 0.096988454 0.096988454; 0.096988454 0.096988454 … 0.096988454 0.096988454;;; … ;;; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0; … ; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0;;; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0; … ; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0;;; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0; … ; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0;;;; 0.68883497 0.5694502 … -0.015367222 -0.38314348; 0.6307154 0.51662284 … 0.70227534 0.45780477; … ; 0.16192031 -0.06471239 … 0.10034483 -0.040765632; 0.89103013 0.51932234 … 0.5314537 0.19592425;;; 1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0; … ; 1.0 1.0 … 1.0 1.0

In [10]:
size(y_img)

(64, 64, 1, 100)

In [11]:
dataloader = DataLoader(mapobs(dev, (x_img, y_img)); batchsize=64, shuffle = true, partial=false)

1-element DataLoader(::MLUtils.MappedData{:auto, ReactantDevice{Missing, Missing, Missing, Missing, Union{}}, Tuple{Array{Float32, 4}, Array{Float32, 4}}}, shuffle=true, batchsize=64, partial=false)
  with first element:
  (64×64×34×64 ConcretePJRTArray{Float32,4}, 64×64×1×64 ConcretePJRTArray{Float32,4},)

In [12]:
dataloader =  load_images(64,im_path, 100, forward, 64,emb_dim,dev)
loadimgs() = load_images(64,im_path, 100, forward, 64,emb_dim,dev)

loadimgs (generic function with 1 method)

In [13]:
#_, loss, _, train_state = Training.single_train_step!(AutoEnzyme(), MSELoss(), (xᵢ, yᵢ), train_state)

In [14]:
test_data = dataloader

2-element DataLoader(::MLUtils.MappedData{:auto, ReactantDevice{Missing, Missing, Missing, Missing, Union{}}, Tuple{Array{Float32, 4}, Array{Float32, 4}}}, shuffle=true, batchsize=64)
  with first element:
  (64×64×34×64 ConcretePJRTArray{Float32,4}, 64×64×1×64 ConcretePJRTArray{Float32,4},)

In [23]:
for (i, data) in enumerate(dataloader)
    print(typeof(data))
    #_, loss, _, train_state = train!(data, train_state)
    #println("$i : $loss")
end

Tuple{ConcretePJRTArray{Float32, 4, 1}, ConcretePJRTArray{Float32, 4, 1}}Tuple{ConcretePJRTArray{Float32, 4, 1}, ConcretePJRTArray{Float32, 4, 1}}

In [16]:
function train_loop!(train_state,loadimgs,epochs = 1; log_every = 10)
    losses = Float32[]
    for epoch in 1:epochs
        t = now(); println("Starting Training for epoch $epoch ...   $t\n")
        dataloader = loadimgs()
        epoch_loss = 0.0f0
        num_batches = length(dataloader)
        for (i, data) in enumerate(dataloader)
            _, loss, _, train_state = train!(data, train_state)
            epoch_loss += loss
            push!(losses, loss)
            print("-")
            if i % log_every == 0
                t = now()
                print(
                    " Batch $i/$num_batches | " *
                    "Loss = $(round(loss; sigdigits=4))" *
                    " Time: $t \n"
                )
            end
        end
        avg_epoch_loss = epoch_loss / num_batches
        t = now()
        ps = train_state.ps
        st = train_state.st
        @save "snapshots/ps$(avg_epoch_loss)_$t.jld2" ps
        @save "ps_latestvn.jld2" ps
        @save "snapshots/st$(avg_epoch_loss)_$t.jld2" st
        @save "st_latestvn.jld2" st
        print(
            "\n=== Epoch $epoch finished | " *
            "avg loss = $(round(avg_epoch_loss; sigdigits=5)) | "*
            " Time: $t" *
            " ===\n"
        )
    end
    return train_state, losses
end

train_loop! (generic function with 2 methods)

In [17]:
#train_state, losses = train_loop!(train_state,loadimgs)